# 🏥 IKNL Web Scraper

**Project:** The Insights Lab — CSN Patient Navigator Knowledge Base  
**Course:** LSE Data Analytics Career Accelerator — Employer Project  
**Author:** Latafat Khankishiyeva  
**Target:** [iknl.nl](https://iknl.nl) — Integraal Kankercentrum Nederland (Dutch national oncology institute)

---

## 🎯 Purpose

Produce a clean, structured Dutch dataset from IKNL — **scope: scraping & structuring only**. 

## 🏗️ Output structure — Seamus-style, one row per cancer type

Each row is one cancer; each thematic column is the Dutch text scraped from a different sub-page of IKNL. This shape mirrors Seamus's `kanker_nl_master_table.csv` so the two files concatenate cleanly later.

| Column | What it holds | IKNL sub-page |
|---|---|---|
| `Cancer Type` | Dutch cancer name (e.g. *Borstkanker*) | — |
| `Source URL` | URL to the cancer's overview page | `/kankersoorten/{slug}` |
| `General Description` | What the cancer is (intro paragraphs) | `/kankersoorten/{slug}` |
| `Decision-making` | Shared-decision-making content | `/kankersoorten/{slug}/besluitvorming` |
| `Treatment` | Treatment-pathway content | `/kankersoorten/{slug}/behandeling` |
| `Statistics & Survival` | Incidence / survival figures | `/kankersoorten/{slug}/cijfers/over-de-cijfers` |
| `Life after cancer` | After-care, quality of life | `/kankersoorten/{slug}/leven-na-kanker` |
| `Palliative phase` | Palliative-care information | `/kankersoorten/{slug}/palliatieve-fase` |
| `Research` | Research / registrations / studies | `/kankersoorten/{slug}/onderzoek` |
| `Scrape Date` | When this row was captured (`YYYY-MM-DD`) | — |
| `Content Hash` | Fingerprint of all Dutch text (for refresh logic) | — |
| `Hyperlinks Preserved` | All in-page links from the overview, pipe-separated | — |

## 🧭 How these columns map to CSN's 6 care-pathway topics

| IKNL column | CSN topic it helps answer |
|---|---|
| Decision-making + Treatment | **Treatment Decisions** ✅ |
| Life after cancer + Palliative phase | **Additional Care & Support** ✅ |
| Statistics & Survival | Contextual data for **Waiting Times** / **Treatment Decisions** |
| General Description + Research | Background that supports **Communication & Understanding** |

## ✅ Design choices to minimise `N/A`s

1. Sub-pages were chosen because **≥60% of cancers on IKNL actually have them** (verified by probing all 23 cancer pages).
2. If a specific sub-page is missing for a cancer, the cell records `N/A` and the row's `Notes` field flags it — so the team can see *why* a cell is empty, not just *that* it is.
3. Local HTML caching means re-running the notebook to refine extraction takes seconds, not minutes.
4. Polite scraping (1.5s delay, custom User-Agent) — won't get the team banned.

## 📤 Deliverable

- **`iknl_master_table.csv`** — UTF-8 CSV, ready for Step 3 (translation)
- **`iknl_master_table.xlsx`** — Excel equivalent for the team to review easily


## 📦 Step 0 — Imports

In [1]:
# Run once if needed:
# !pip install requests beautifulsoup4 pandas openpyxl

import os
import re
import time
import hashlib
import logging
from datetime import datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

print('✅ Imports OK')

✅ Imports OK


## ⚙️ Step 1 — Configuration

In [2]:
# ── Source identity ──────────────────────────────────────────────────
BASE_URL    = 'https://iknl.nl'
INDEX_URL   = 'https://iknl.nl/kankersoorten'
SOURCE_ORG  = 'IKNL (Integraal Kankercentrum Nederland)'

# ── Scraper behaviour ────────────────────────────────────────────────
HEADERS = {
    'User-Agent': (
        'TheInsightsLab-LSE-Research-Bot/1.0 '
        '(Academic project for Cancer Support Netherlands)'
    )
}
REQUEST_DELAY_SECONDS = 1.5    # be polite to iknl.nl
REQUEST_TIMEOUT       = 15
CACHE_DIR             = 'raw_html_cache_iknl'
ERROR_LOG             = 'iknl_scrape_errors.log'
OUTPUT_CSV            = 'iknl_master_table.csv'
OUTPUT_XLSX           = 'iknl_master_table.xlsx'

# ── The 6 IKNL sub-pages we scrape per cancer ────────────────────────
# Format: column_name → sub_url_segment (relative to /kankersoorten/{slug}/)
# Empty string '' means the overview page itself.
SUB_PAGES = {
    'General Description':    '',                          # 96%+ cancers
    'Decision-making':        'besluitvorming',            # 17% — only common cancers
    'Treatment':              'behandeling',               # 61%
    'Statistics & Survival':  'cijfers/over-de-cijfers',   # 96%
    'Life after cancer':      'leven-na-kanker',           # 65%
    'Palliative phase':       'palliatieve-fase',          # 83%
    'Research':               'onderzoek',                 # 87%
}

os.makedirs(CACHE_DIR, exist_ok=True)
logging.basicConfig(filename=ERROR_LOG, level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s')
print('✅ Configuration loaded')
print(f'   Sub-pages per cancer: {len(SUB_PAGES)}')

✅ Configuration loaded
   Sub-pages per cancer: 7


## 🛠️ Step 2 — Polite fetcher with on-disk HTML cache

The cache lets us re-run the extraction logic without re-hitting iknl.nl (and lets the team re-process offline).

In [3]:
def _safe_filename(url: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', url) + '.html'


def fetch(url: str) -> str | None:
    """Fetch a URL with on-disk caching. Returns HTML, or None on failure."""
    cache_path = os.path.join(CACHE_DIR, _safe_filename(url))
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8') as f:
            return f.read()
    try:
        r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        # 404 / 5xx is not necessarily an error — store the empty page
        if r.status_code != 200:
            logging.info(f'NON_200 | {url} | {r.status_code}')
            time.sleep(REQUEST_DELAY_SECONDS)
            return None
        time.sleep(REQUEST_DELAY_SECONDS)
        with open(cache_path, 'w', encoding='utf-8') as f:
            f.write(r.text)
        return r.text
    except requests.RequestException as e:
        logging.error(f'FETCH_FAIL | {url} | {e}')
        return None

print('✅ Fetcher ready')

✅ Fetcher ready


## 🔎 Step 3 — Discover all cancer-type slugs on IKNL

Crawls the `/kankersoorten` landing page and returns `{slug: display_name}`. Section-seeded crawling, not a hard-coded list — if IKNL adds a new cancer page tomorrow, we pick it up automatically.

In [4]:
def discover_cancers() -> dict[str, str]:
    """Return {slug: cancer_display_name} for every cancer on IKNL."""
    html = fetch(INDEX_URL)
    if not html:
        return {}
    soup = BeautifulSoup(html, 'html.parser')
    main = soup.find('main') or soup
    out: dict[str, str] = {}
    for a in main.find_all('a', href=True):
        href = a['href'].strip()
        # Direct child only — /kankersoorten/borstkanker, not deeper
        if href.startswith('/kankersoorten/') and href.count('/') == 2:
            slug = href.rsplit('/', 1)[-1]
            name = a.get_text(strip=True)
            if name and len(name) < 80 and slug not in out:
                out[slug] = name
    return out


cancers = discover_cancers()
print(f'✅ Discovered {len(cancers)} cancer types on IKNL')
print()
for slug, name in list(cancers.items())[:10]:
    print(f'  • {name:<35} → /kankersoorten/{slug}')
print('  ...')

✅ Discovered 23 cancer types on IKNL

  • Baarmoederhalskanker                → /kankersoorten/baarmoederhalskanker
  • Baarmoederkanker                    → /kankersoorten/baarmoederkanker
  • Blaaskanker                         → /kankersoorten/blaaskanker
  • Borstkanker                         → /kankersoorten/borstkanker
  • Bot- en wekedelenkanker             → /kankersoorten/bot-en-wekedelenkanker
  • Darmkanker                          → /kankersoorten/darmkanker
  • Eierstokkanker                      → /kankersoorten/eierstokkanker
  • Hemato-oncologie                    → /kankersoorten/hemato-oncologie
  • Hersentumoren                       → /kankersoorten/hersentumoren
  • Hoofd-halskanker                    → /kankersoorten/hoofd-halskanker
  ...


## 🧹 Step 4 — Extract clean Dutch text from one IKNL page

Strategy:

1. Find the `<main>` tag — IKNL puts all real content there.
2. Strip nav / aside / footer / forms before extracting.
3. Walk `<h2>`, `<h3>`, `<p>`, `<li>` in document order so we preserve the *meaning* of the page (heading → paragraph relationships) rather than producing a flat text blob.
4. Collapse whitespace and strip non-breaking spaces.

In [5]:
def _clean(text: str) -> str:
    return re.sub(r'\s+', ' ', text).replace('\xa0', ' ').strip()


def extract_dutch_text(url: str) -> str:
    """Scrape one IKNL page and return its substantive Dutch body text.

    Returns 'N/A' if the page doesn't exist, has no <main>, or is empty.
    """
    html = fetch(url)
    if not html:
        return 'N/A'
    soup = BeautifulSoup(html, 'html.parser')
    main = soup.find('main')
    if main is None:
        return 'N/A'
    body_lower = main.get_text().lower()
    if 'niet beschikbaar' in body_lower and len(body_lower) < 600:
        return 'N/A'
    # Strip noise inside <main>
    for tag in main.find_all(['nav', 'aside', 'footer', 'form']):
        tag.decompose()
    # Strip IKNL's collapsible sub-navigation (breadcrumb-style links)
    for div in main.find_all('div', class_=re.compile(r'coloured-expandable-section')):
        div.decompose()
    # Walk content elements in order
    parts: list[str] = []
    skip_section_titles = {
        'nieuws', 'medewerkers', 'contact', 'navigatie',
        'lees meer', 'gerelateerde links', 'rapportages',
    }
    for el in main.find_all(['h2', 'h3', 'p', 'li']):
        if el.name in ('h2', 'h3'):
            heading = _clean(el.get_text())
            if heading and heading.lower() not in skip_section_titles:
                parts.append(f'\n## {heading}')
        else:
            txt = _clean(el.get_text())
            # Skip very short junk (single labels, navigation crumbs)
            if txt and len(txt) > 15:
                parts.append(txt)
    body = ' '.join(parts).strip()
    return body if len(body) >= 50 else 'N/A'


def extract_hyperlinks(url: str, max_links: int = 25) -> str:
    """Return up to 25 pipe-separated `anchor -> full_url` strings from <main>."""
    html = fetch(url)
    if not html:
        return ''
    soup = BeautifulSoup(html, 'html.parser')
    main = soup.find('main')
    if not main:
        return ''
    for tag in main.find_all(['nav', 'aside', 'footer']):
        tag.decompose()
    for div in main.find_all('div', class_=re.compile(r'coloured-expandable-section')):
        div.decompose()
    seen, links = set(), []
    for a in main.find_all('a', href=True):
        anchor = _clean(a.get_text())
        if not anchor or len(anchor) < 2:
            continue
        full = urljoin(BASE_URL, a['href'].strip())
        item = f'{anchor} -> {full}'
        if item not in seen:
            seen.add(item)
            links.append(item)
            if len(links) >= max_links:
                break
    return ' | '.join(links)


# Sanity test
sample = extract_dutch_text('https://iknl.nl/kankersoorten/borstkanker/behandeling')
print(f'Sample (borstkanker /behandeling) — {len(sample)} chars:')
print(sample[:400] + '…' if len(sample) > 400 else sample)

Sample (borstkanker /behandeling) — 1339 chars:
De prognose voor borstkanker is in de afgelopen decennia aanzienlijk verbeterd. Dit is het resultaat van eerdere diagnoses en effectievere behandelingen. Tegelijkertijd groeit de aandacht voor de late gevolgen van de behandeling. De focus verschuift steeds meer naar het optimaliseren van de kwaliteit van leven na de behandeling. Er is een duidelijke trend zichtbaar richting minder intensieve thera…


## 🏗️ Step 5 — Build one row per cancer

For each cancer, scrape every sub-page in `SUB_PAGES` and put the resulting Dutch text into a separate column. Also record metadata so the row is fully traceable.

In [6]:
def build_row(slug: str, cancer_name: str) -> dict:
    """Return one full row of the master table for a given cancer."""
    overview_url = f'{BASE_URL}/kankersoorten/{slug}'

    row = {
        'Cancer Type':         cancer_name,
        'Source URL':          overview_url,
        'Source Organisation': SOURCE_ORG,
    }

    # Scrape each themed sub-page
    missing_columns = []
    for col, sub in SUB_PAGES.items():
        sub_url = f'{overview_url}/{sub}' if sub else overview_url
        text = extract_dutch_text(sub_url)
        row[col] = text
        if text == 'N/A':
            missing_columns.append(col)

    # Audit fields
    row['Scrape Date'] = datetime.utcnow().strftime('%Y-%m-%d')
    # Hash of all populated Dutch text — supports refresh logic
    all_dutch = ' '.join(row[c] for c in SUB_PAGES if row[c] != 'N/A')
    row['Content Hash'] = hashlib.md5(all_dutch.encode('utf-8')).hexdigest()[:12]
    row['Body Length Chars'] = len(all_dutch)
    row['Hyperlinks Preserved'] = extract_hyperlinks(overview_url)

    # Notes — record which sub-pages were missing, for transparency
    if missing_columns:
        row['Notes'] = 'Sub-pages missing on IKNL: ' + ', '.join(missing_columns)
    else:
        row['Notes'] = 'All sub-pages scraped successfully'
    return row


# Test on one cancer
demo = build_row('borstkanker', 'Borstkanker')
for k, v in demo.items():
    if isinstance(v, str) and len(v) > 100:
        print(f'  {k:<25}: ({len(v)} chars) {v[:80]}…')
    else:
        print(f'  {k:<25}: {v}')

  Cancer Type              : Borstkanker
  Source URL               : https://iknl.nl/kankersoorten/borstkanker
  Source Organisation      : IKNL (Integraal Kankercentrum Nederland)
  General Description      : (1930 chars) Borstkanker is wereldwijd een veelvoorkomende vorm van kanker die met name in we…
  Decision-making          : (3991 chars) Om het proces rondom het multidisciplinair overleg (mdo) verder te optimaliseren…
  Treatment                : (1339 chars) De prognose voor borstkanker is in de afgelopen decennia aanzienlijk verbeterd. …
  Statistics & Survival    : (3688 chars) In de Nederlandse Kankerregistratie (NKR) worden gegevens over diagnostiek en pr…
  Life after cancer        : (1640 chars) Veel vrouwen ervaren na de behandeling van borstkanker langdurige gezondheidspro…
  Palliative phase         : (3483 chars) Wanneer genezing van borstkanker niet meer mogelijk is, kunnen zorgprofessionals…
  Research                 : (3404 chars) De NKR-gegevens vormen een belan

/var/folders/c4/rh7_h5xx6y9d2lwq8q6m80lh0000gn/T/ipykernel_11865/1332348696.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row['Scrape Date'] = datetime.utcnow().strftime('%Y-%m-%d')


## 🚀 Step 6 — Run the full scrape

23 cancers × 7 sub-pages × 1.5 s polite delay = roughly 5 minutes the first time (cached afterwards).

In [ ]:
rows = []
for i, (slug, name) in enumerate(cancers.items(), 1):
    print(f'[{i:>2}/{len(cancers)}]  {name:<35} → /kankersoorten/{slug}')
    rows.append(build_row(slug, name))

df = pd.DataFrame(rows)
print(f'\n✅ Scraped {len(df)} cancers × {len(df.columns)} columns')

[ 1/23]  Baarmoederhalskanker                → /kankersoorten/baarmoederhalskanker


/var/folders/c4/rh7_h5xx6y9d2lwq8q6m80lh0000gn/T/ipykernel_11865/1332348696.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row['Scrape Date'] = datetime.utcnow().strftime('%Y-%m-%d')


[ 2/23]  Baarmoederkanker                    → /kankersoorten/baarmoederkanker
[ 3/23]  Blaaskanker                         → /kankersoorten/blaaskanker
[ 4/23]  Borstkanker                         → /kankersoorten/borstkanker
[ 5/23]  Bot- en wekedelenkanker             → /kankersoorten/bot-en-wekedelenkanker
[ 6/23]  Darmkanker                          → /kankersoorten/darmkanker
[ 7/23]  Eierstokkanker                      → /kankersoorten/eierstokkanker
[ 8/23]  Hemato-oncologie                    → /kankersoorten/hemato-oncologie
[ 9/23]  Hersentumoren                       → /kankersoorten/hersentumoren
[10/23]  Hoofd-halskanker                    → /kankersoorten/hoofd-halskanker


## 🔍 Step 7 — Quality check

Show how complete each thematic column is. Anything below ~60% fill rate is a column where IKNL simply doesn't publish that information consistently — and that's a useful **finding** for Makhotso's gap analysis (Step 6).

In [ ]:
print('─ Per-column fill rate (substantive content vs N/A) ─')
for col in SUB_PAGES:
    filled = (df[col] != 'N/A').sum()
    pct = filled / len(df)
    bar = '█' * int(pct * 25) + '·' * (25 - int(pct * 25))
    print(f'  {col:<25} {bar} {filled:>2}/{len(df)} ({pct:.0%})')

print('\n─ Body length distribution (chars of Dutch text per cancer) ─')
print(df['Body Length Chars'].describe().astype(int).to_string())

print('\n─ 5 most content-rich cancers ─')
top5 = df.nlargest(5, 'Body Length Chars')[['Cancer Type', 'Body Length Chars']]
print(top5.to_string(index=False))

print('\n─ 5 thinnest cancers (likely rare cancer types) ─')
bot5 = df.nsmallest(5, 'Body Length Chars')[['Cancer Type', 'Body Length Chars', 'Notes']]
print(bot5.to_string(index=False, max_colwidth=60))

─ Per-column fill rate (substantive content vs N/A) ─
  General Description       █████████████████████████ 23/23 (100%)
  Decision-making           ███████████████████████·· 22/23 (96%)
  Treatment                 ███████████████████████·· 22/23 (96%)
  Statistics & Survival     █████████████████████████ 23/23 (100%)
  Life after cancer         █████████████████████████ 23/23 (100%)
  Palliative phase          █████████████████████████ 23/23 (100%)
  Research                  █████████████████████████ 23/23 (100%)

─ Body length distribution (chars of Dutch text per cancer) ─
count       23
mean     15331
std       7020
min       4802
25%      10815
50%      14109
75%      18704
max      37708

─ 5 most content-rich cancers ─
    Cancer Type  Body Length Chars
     Longkanker              37708
 Prostaatkanker              24874
Zeldzame kanker              24590
 Eierstokkanker              20682
    Borstkanker              19481

─ 5 thinnest cancers (likely rare cancer types) ─
  

## 💾 Step 8 — Save the master table (CSV + Excel)

Both files use the same column order so the team can use either one interchangeably.

In [ ]:
# Canonical column order — matches the structure of Seamus's kanker_nl table
COLUMN_ORDER = [
    'Cancer Type',
    'Source URL',
    'Source Organisation',
    'General Description',
    'Decision-making',
    'Treatment',
    'Statistics & Survival',
    'Life after cancer',
    'Palliative phase',
    'Research',
    'Scrape Date',
    'Content Hash',
    'Body Length Chars',
    'Hyperlinks Preserved',
    'Notes',
]
df = df[COLUMN_ORDER]

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
df.to_excel(OUTPUT_XLSX, index=False)

print(f'✅ Saved {len(df)} rows × {len(df.columns)} columns')
print(f'   • {OUTPUT_CSV}')
print(f'   • {OUTPUT_XLSX}')

✅ Saved 23 rows × 15 columns
   • iknl_master_table.csv
   • iknl_master_table.xlsx


## 👀 Step 9 — Preview the output

A quick look at the first 6 rows so you can sanity-check before handing off to the translation step.

In [ ]:
pd.set_option('display.max_colwidth', 60)
df.head(6)[['Cancer Type', 'General Description', 'Treatment', 'Life after cancer', 'Body Length Chars']]

,Cancer Type,General Description,Treatment,Life after cancer,Body Length Chars
0,Baarmoederhalskanker,Per jaar krijgen rond de 900 vrouwen in Nederland voor h...,De behandeling van baarmoederhalskanker ofwel cervixcarc...,"De pagina die u heeft opgevraagd, is niet gevonden. \n##...",15824
1,Baarmoederkanker,Baarmoederkanker is de meest voorkomende gynaecologische...,"De pagina die u heeft opgevraagd, is niet gevonden. \n##...","De pagina die u heeft opgevraagd, is niet gevonden. \n##...",8566
2,Blaaskanker,Jaarlijks krijgen ongeveer 7.000 mensen voor het eerst d...,De grootste groep patiënten heeft een niet-spierinvasiev...,Na de behandeling van kanker kunnen patiënten late gevol...,15739
3,Borstkanker,Borstkanker is wereldwijd een veelvoorkomende vorm van k...,De prognose voor borstkanker is in de afgelopen decennia...,Veel vrouwen ervaren na de behandeling van borstkanker l...,19481
4,Bot- en wekedelenkanker,"Kwaadaardige bot- en wekedelentumoren (sarcomen), vormen...",In de Nederlandse Kankerregistratie worden gegevens verz...,Bijna 800.000 mensen leven met of na kanker en een aanzi...,13405
5,Darmkanker,Darmkanker is een veel voorkomende vorm van kanker in Ne...,"De pagina die u heeft opgevraagd, is niet gevonden. \n##...",Bijna 800.000 mensen leven met of na kanker en een aanzi...,13000


---

## 📤 Hand-off note

**Status:** Step 2 (scraping & structuring) complete for IKNL. Output is the canonical CSV/Excel pair.

**Next step (Step 3 — Translation, my own next task):**
- Translate the 6 Dutch content columns (`General Description`, `Decision-making`, `Treatment`, `Statistics & Survival`, `Life after cancer`, `Palliative phase`, `Research`) into English.
- Use `deep-translator` with sentence-aware chunking for long cells (Google Translate's per-call cap is ~5,000 characters).
- Skip cells where the value is already `N/A` — saves API calls.
- Validate a stratified ~10% sample with the Dutch-speaking acquaintance.

